# Genie Benchmark Runner — Eval API

This notebook uses the **Genie Eval API** (`genie.genie_create_eval_run`) to run benchmark
questions against a Genie space, collect assessment results, and log everything to MLflow.

### What it does

1. Kicks off **N independent eval runs** via the Eval API (each run evaluates all benchmark questions)
2. Polls each run until completion, then fetches per-question results with detailed assessments
3. Aggregates accuracy statistics across all runs
4. Logs results, metrics, and artifacts to MLflow
5. Analyses failure patterns and generates improvement recommendations

### Differences from the original notebook

- **No manual SQL execution or dataset comparison** — the Eval API handles evaluation server-side
  using an LLM judge plus result-set comparison, returning `GOOD` / `BAD` / `NEEDS_REVIEW` assessments
- **Richer failure diagnostics** — the API returns structured `ScoreReason` codes
  (e.g. `LLM_JUDGE_MISSING_OR_INCORRECT_FILTER`, `RESULT_MISSING_ROWS`) for each question

In [ ]:
%pip install databricks-sdk mlflow langchain langchain-databricks langchain-core -qU

dbutils.library.restartPython()

In [ ]:
import json
import time
import pandas as pd
from datetime import datetime
from collections import Counter

import mlflow
from databricks.sdk import WorkspaceClient

## Configuration

Set your Genie space ID and tuning parameters below.

When running inside a Databricks notebook, `WorkspaceClient()` authenticates automatically.
For local execution, set the `DATABRICKS_HOST` and `DATABRICKS_TOKEN` environment variables.

In [ ]:
GENIE_SPACE_ID  = "<your-genie-space-id>"
NUM_RUNS        = 5       # number of independent eval runs to execute
POLL_INTERVAL   = 10      # seconds between status checks
MAX_WAIT_SEC    = 600     # per-run timeout in seconds
MLFLOW_EXPERIMENT_NAME = "/genie-benchmark-runner"

## Initialise Client & Validate Space

In [ ]:
w = WorkspaceClient()

space = w.genie.get_space(
    space_id=GENIE_SPACE_ID,
    include_serialized_space=True,
)

space_config = json.loads(space.serialized_space)
benchmark_questions = space_config.get("benchmarks", {}).get("questions", [])

if not benchmark_questions:
    raise ValueError(
        f"No benchmark questions found in space '{GENIE_SPACE_ID}'. "
        "Add benchmark questions to the Genie space before running this notebook."
    )

print(f"Space:               {space.title}")
print(f"Description:         {space.description or 'N/A'}")
print(f"Warehouse:           {space.warehouse_id}")
print(f"Benchmark questions: {len(benchmark_questions)}")
print(f"Eval runs planned:   {NUM_RUNS}")

print("\n--- Benchmark Questions ---")
for i, q in enumerate(benchmark_questions, 1):
    question_text = " ".join(q.get("question", [])).strip()
    print(f"  {i:>2}. {question_text}")

## Helper Functions

### `run_eval`
Kicks off an eval run via the SDK and polls until it completes or times out.

### `collect_eval_results`
Paginates through all results for a completed eval run and fetches detailed
assessments for each question.

In [ ]:
def run_eval(w, space_id, poll_interval=POLL_INTERVAL, max_wait=MAX_WAIT_SEC):
    """
    Create an eval run for all benchmark questions and poll until done.

    Returns
    -------
    eval_run : GenieEvalRunResponse
    """
    eval_run = w.genie.genie_create_eval_run(space_id=space_id)
    eval_run_id = eval_run.eval_run_id

    deadline = time.time() + max_wait
    while time.time() < deadline:
        eval_run = w.genie.genie_get_eval_run(
            space_id=space_id,
            eval_run_id=eval_run_id,
        )
        status = eval_run.eval_run_status
        status_str = status.value if hasattr(status, "value") else str(status)

        done = eval_run.num_done or 0
        total = eval_run.num_questions or 0
        print(f"    Status: {status_str}  ({done}/{total} questions done)", end="\r")

        if status_str.upper() in ("DONE", "EVALUATION_CANCELLED", "EVALUATION_FAILED", "EVALUATION_TIMEOUT"):
            print()  # newline after \r
            return eval_run

        time.sleep(poll_interval)

    print()
    print(f"    WARNING: Eval run {eval_run_id} timed out after {max_wait}s")
    return eval_run


def collect_eval_results(w, space_id, eval_run_id):
    """
    Fetch all eval results (with details) for a completed run.

    Returns
    -------
    results : list[dict]
        One dict per benchmark question with keys:
        result_id, question, benchmark_question_id, assessment,
        score_reasons, expected_sql, generated_sql, status
    """
    # Paginate through all results
    all_results = []
    page_token = None
    while True:
        resp = w.genie.genie_list_eval_results(
            space_id=space_id,
            eval_run_id=eval_run_id,
            page_size=100,
            page_token=page_token,
        )
        if resp.eval_results:
            all_results.extend(resp.eval_results)
        page_token = resp.next_page_token
        if not page_token:
            break

    # Fetch details for each result
    detailed = []
    for result in all_results:
        try:
            details = w.genie.genie_get_eval_result_details(
                space_id=space_id,
                eval_run_id=eval_run_id,
                result_id=result.result_id,
            )
        except Exception as e:
            print(f"    WARNING: Could not fetch details for result {result.result_id}: {e}")
            details = None

        # Extract assessment
        assessment = None
        score_reasons = []
        expected_sql = None
        generated_sql = None

        if details:
            if details.assessment:
                assessment = details.assessment.value if hasattr(details.assessment, "value") else str(details.assessment)
            if details.assessment_reasons:
                score_reasons = [
                    r.value if hasattr(r, "value") else str(r)
                    for r in details.assessment_reasons
                ]
            # Extract expected response SQL
            if details.expected_response:
                for resp_item in details.expected_response:
                    resp_type = resp_item.response_type
                    resp_type_str = resp_type.value if hasattr(resp_type, "value") else str(resp_type)
                    if resp_type_str == "SQL" and resp_item.response:
                        expected_sql = resp_item.response
                        break
            # Extract actual response SQL
            if details.actual_response:
                for resp_item in details.actual_response:
                    resp_type = resp_item.response_type
                    resp_type_str = resp_type.value if hasattr(resp_type, "value") else str(resp_type)
                    if resp_type_str == "SQL" and resp_item.response:
                        generated_sql = resp_item.response
                        break

        status = result.status
        status_str = status.value if hasattr(status, "value") else str(status)

        detailed.append({
            "result_id":             result.result_id,
            "question":              result.question or "",
            "benchmark_question_id": result.benchmark_question_id,
            "assessment":            assessment,
            "score_reasons":         score_reasons,
            "expected_sql":          expected_sql,
            "generated_sql":         generated_sql,
            "status":                status_str,
        })

    return detailed

## Run Benchmark Evaluations

Execute **NUM_RUNS** independent eval runs. Each run sends all benchmark questions to
the Genie Eval API, which handles question submission, SQL generation, execution,
and assessment server-side.

In [ ]:
all_run_results = []  # list of (eval_run_response, detailed_results)
raw_results = []       # flat list for the results DataFrame

for run_idx in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  Eval Run {run_idx} / {NUM_RUNS}")
    print(f"{'='*60}")

    eval_run = run_eval(w, GENIE_SPACE_ID)

    status_str = eval_run.eval_run_status.value if hasattr(eval_run.eval_run_status, "value") else str(eval_run.eval_run_status)
    print(f"  Run completed: status={status_str}")
    print(f"    correct={eval_run.num_correct}  done={eval_run.num_done}  "
          f"needs_review={eval_run.num_needs_review}  total={eval_run.num_questions}")

    # Collect detailed results
    detailed = collect_eval_results(w, GENIE_SPACE_ID, eval_run.eval_run_id)
    all_run_results.append((eval_run, detailed))

    # Print per-question results
    for q_idx, r in enumerate(detailed, 1):
        icon = "PASS" if r["assessment"] == "GOOD" else "FAIL" if r["assessment"] == "BAD" else "REVIEW"
        reasons_str = ", ".join(r["score_reasons"][:3]) if r["score_reasons"] else ""
        print(f"    Q{q_idx:>2}. [{icon:>6}] {r['question'][:70]}")
        if reasons_str:
            print(f"               Reasons: {reasons_str}")

    # Flatten into raw_results
    for r in detailed:
        passed = r["assessment"] == "GOOD"
        raw_results.append({
            "run_index":             run_idx,
            "eval_run_id":           eval_run.eval_run_id,
            "question":              r["question"],
            "benchmark_question_id": r["benchmark_question_id"],
            "assessment":            r["assessment"],
            "score_reasons":         "; ".join(r["score_reasons"]) if r["score_reasons"] else "",
            "expected_sql":          r["expected_sql"],
            "generated_sql":         r["generated_sql"],
            "passed":                passed,
            "status":                r["status"],
        })

    correct = eval_run.num_correct or 0
    total = eval_run.num_questions or 0
    pct = (correct / total * 100) if total > 0 else 0
    print(f"\n  Run {run_idx} accuracy: {correct}/{total}  ({pct:.1f}%)")

print("\nAll eval runs complete.")

## Results DataFrame

Full results table with one row per (run, question).

| Column | Description |
|--------|-------------|
| `run_index` | Which run (1–NUM_RUNS) |
| `question` | The benchmark question text |
| `assessment` | `GOOD`, `BAD`, or `NEEDS_REVIEW` |
| `score_reasons` | Structured failure reasons from the Eval API |
| `expected_sql` | Expected SQL from the benchmark |
| `generated_sql` | SQL produced by Genie |
| `passed` | Whether assessment was `GOOD` |

In [ ]:
results_df = pd.DataFrame(raw_results)

results_df = results_df[[
    "run_index", "question", "assessment", "score_reasons",
    "expected_sql", "generated_sql", "passed", "status",
]]

print(f"Total rows: {len(results_df)}  ({NUM_RUNS} runs \u00d7 {len(benchmark_questions)} questions)")

try:
    display(results_df)  # noqa: F821
except NameError:
    print(results_df.to_string())

## Summary Statistics

Per-run accuracy and overall accuracy averaged across all runs.

In [ ]:
per_run_accuracy = (
    results_df.groupby("run_index")["passed"]
    .agg(passed_count="sum", total="count")
    .assign(accuracy_pct=lambda df: df["passed_count"] / df["total"] * 100)
    .reset_index()
)

overall_accuracy = per_run_accuracy["accuracy_pct"].mean()

# Assessment distribution across all results
assessment_counts = results_df["assessment"].value_counts().to_dict()

print(f"Space:            {space.title}")
print(f"Benchmark count:  {len(benchmark_questions)}")
print(f"Runs:             {NUM_RUNS}")
print()
print("Assessment distribution:")
for assess, count in sorted(assessment_counts.items()):
    print(f"  {assess:15s} {count:>4}")
print()
print("Per-run accuracy:")
for _, row in per_run_accuracy.iterrows():
    bar = '\u2588' * int(row['accuracy_pct'] / 5) + '\u2591' * (20 - int(row['accuracy_pct'] / 5))
    print(f"  Run {int(row['run_index'])}: {bar} {row['accuracy_pct']:5.1f}%  "
          f"({int(row['passed_count'])}/{int(row['total'])})")

bar = '\u2588' * int(overall_accuracy / 5) + '\u2591' * (20 - int(overall_accuracy / 5))
print(f"\n  Overall: {bar} {overall_accuracy:5.1f}%")

try:
    display(per_run_accuracy)  # noqa: F821
except NameError:
    print(per_run_accuracy.to_string())

## MLflow Run

Log all results to MLflow:

| Logged Item | Type | Description |
|-------------|------|-------------|
| `space_id` | param | Genie space ID |
| `space_name` | param | Human-readable space name |
| `num_benchmarks` | param | Number of benchmark questions |
| `num_runs` | param | Number of eval runs executed |
| `evaluation_method` | param | `genie_eval_api` |
| `overall_accuracy` | metric | Average accuracy across all runs |
| `run_N_accuracy` | metric | Per-run accuracy |
| `q_N_accuracy` | metric | Per-question accuracy across runs |
| `benchmark_results.csv` | artifact | Full results table |
| `per_question_accuracy.csv` | artifact | Per-question aggregated stats |
| `score_reasons_summary.json` | artifact | Frequency of each score reason |
| `genie_space_config.json` | artifact | Snapshot of the Genie space config |

In [ ]:
import tempfile
import os

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{space.title.replace(' ', '_')}_{run_timestamp}"

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=run_name) as mlflow_run:

    mlflow.log_params({
        "space_id":          GENIE_SPACE_ID,
        "space_name":        space.title,
        "num_benchmarks":    len(benchmark_questions),
        "num_runs":          NUM_RUNS,
        "evaluation_method": "genie_eval_api",
    })

    # --- Per-run accuracy metrics ---
    for _, row in per_run_accuracy.iterrows():
        run_num = int(row["run_index"])
        mlflow.log_metric(f"run_{run_num}_accuracy", round(row["accuracy_pct"], 2))

    # --- Per-question accuracy metrics ---
    per_question_accuracy = (
        results_df.groupby("question")["passed"]
        .agg(passes="sum", total="count")
        .assign(accuracy_pct=lambda df: (df["passes"] / df["total"] * 100).round(2))
        .sort_index()
        .reset_index()
    )
    per_question_accuracy.insert(0, "q_index", range(1, len(per_question_accuracy) + 1))

    for _, row in per_question_accuracy.iterrows():
        mlflow.log_metric(f"q_{int(row['q_index'])}_accuracy", row["accuracy_pct"])

    # --- Overall accuracy ---
    mlflow.log_metric("overall_accuracy", round(overall_accuracy, 2))

    # --- Assessment distribution ---
    for assess, count in assessment_counts.items():
        mlflow.log_metric(f"assessment_{assess.lower()}", count)

    # --- Save artifacts ---
    with tempfile.TemporaryDirectory() as tmpdir:
        csv_path = os.path.join(tmpdir, "benchmark_results.csv")
        results_df.to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path)

        pq_path = os.path.join(tmpdir, "per_question_accuracy.csv")
        per_question_accuracy.to_csv(pq_path, index=False)
        mlflow.log_artifact(pq_path)

        # Score reasons frequency
        all_reasons = []
        for reasons_str in results_df["score_reasons"]:
            if reasons_str:
                all_reasons.extend([r.strip() for r in reasons_str.split(";") if r.strip()])
        reason_counts = dict(Counter(all_reasons).most_common())

        reasons_path = os.path.join(tmpdir, "score_reasons_summary.json")
        with open(reasons_path, "w") as f:
            json.dump(reason_counts, f, indent=2)
        mlflow.log_artifact(reasons_path)

        space_json_path = os.path.join(tmpdir, "genie_space_config.json")
        with open(space_json_path, "w") as f:
            json.dump(space_config, f, indent=2)
        mlflow.log_artifact(space_json_path)

    run_id = mlflow_run.info.run_id

print(f"MLflow run complete.")
print(f"  Run name:         {run_name}")
print(f"  Run ID:           {run_id}")
print(f"  Experiment:       {MLFLOW_EXPERIMENT_NAME}")
print(f"  Overall accuracy: {overall_accuracy:.1f}%")
print()
print("Per-run metrics logged:")
for _, row in per_run_accuracy.iterrows():
    print(f"  run_{int(row['run_index'])}_accuracy = {row['accuracy_pct']:.2f}%")
print()
print("Per-question metrics logged:")
for _, row in per_question_accuracy.iterrows():
    print(f"  q_{int(row['q_index'])}_accuracy = {row['accuracy_pct']:.2f}%  \u2014 {row['question'][:60]}")
print(f"\n  overall_accuracy = {overall_accuracy:.2f}%")

## Benchmark Analysis & Recommendations

Analyse failure patterns using the structured `ScoreReason` codes from the Eval API,
then feed the analysis into an LLM to generate actionable improvement recommendations.

In [ ]:
# ------------------------------------------------------------------
# 1. Per-question statistics
# ------------------------------------------------------------------
question_stats = (
    results_df.groupby("question")
    .agg(
        total_runs=("passed", "count"),
        passes=("passed", "sum"),
        failures=("passed", lambda s: (~s).sum()),
    )
    .assign(pass_rate=lambda df: (df["passes"] / df["total_runs"] * 100).round(1))
    .sort_values("pass_rate")
    .reset_index()
)

# ------------------------------------------------------------------
# 2. Score reason breakdown (from the Eval API)
# ------------------------------------------------------------------
all_reasons = []
for reasons_str in results_df["score_reasons"]:
    if reasons_str:
        all_reasons.extend([r.strip() for r in reasons_str.split(";") if r.strip()])

reason_counts = Counter(all_reasons)
failure_summary = dict(reason_counts.most_common())

# ------------------------------------------------------------------
# 3. Build per-question failure detail
# ------------------------------------------------------------------
failed_rows = results_df[~results_df["passed"]].copy()

question_details = []
for _, row in question_stats.iterrows():
    q = row["question"]
    q_failures = failed_rows[failed_rows["question"] == q]

    # Collect unique score reasons for this question
    q_reasons = set()
    for reasons_str in q_failures["score_reasons"]:
        if reasons_str:
            q_reasons.update(r.strip() for r in reasons_str.split(";") if r.strip())

    sample_expected = results_df.loc[results_df["question"] == q, "expected_sql"].iloc[0]
    sample_generated = q_failures["generated_sql"].dropna().head(1).tolist()

    detail = f"  Q: {q}\n"
    detail += f"     Pass rate: {row['pass_rate']}% ({int(row['passes'])}/{int(row['total_runs'])})\n"
    if sample_expected:
        detail += f"     Expected SQL: {sample_expected[:200]}\n"
    if sample_generated:
        detail += f"     Sample generated SQL: {sample_generated[0][:200]}\n"
    if q_reasons:
        detail += f"     Score reasons: {', '.join(sorted(q_reasons))}\n"
    question_details.append(detail)

per_question_text = "\n".join(question_details)

# ------------------------------------------------------------------
# 4. Consistency buckets
# ------------------------------------------------------------------
always_fail    = question_stats[question_stats["pass_rate"] == 0]["question"].tolist()
sometimes_fail = question_stats[
    (question_stats["pass_rate"] > 0) & (question_stats["pass_rate"] < 100)
]["question"].tolist()
always_pass    = question_stats[question_stats["pass_rate"] == 100]["question"].tolist()

consistency_text = (
    f"Always pass ({len(always_pass)}): {', '.join(always_pass[:5])}"
    + (" ..." if len(always_pass) > 5 else "") + "\n"
    f"Intermittent ({len(sometimes_fail)}): {', '.join(sometimes_fail[:5])}"
    + (" ..." if len(sometimes_fail) > 5 else "") + "\n"
    f"Always fail ({len(always_fail)}): {', '.join(always_fail[:5])}"
    + (" ..." if len(always_fail) > 5 else "")
)

# ------------------------------------------------------------------
# 5. Print diagnostic summary
# ------------------------------------------------------------------
print("=== Score Reason Breakdown ===")
for reason, count in reason_counts.most_common():
    print(f"  {reason:55s} {count}")
print(f"\n  Total failures: {len(failed_rows)} / {len(results_df)}")

print(f"\n=== Question Consistency ===")
print(f"  Always pass:    {len(always_pass)}")
print(f"  Intermittent:   {len(sometimes_fail)}")
print(f"  Always fail:    {len(always_fail)}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_databricks import ChatDatabricks

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0)

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at optimising Databricks AI/BI Genie spaces. "
     "You will be given benchmark results from a Genie space — including structured "
     "score reasons from the Genie Eval API — and your job is to analyse the failure "
     "patterns and produce specific, actionable recommendations to improve the space's "
     "accuracy. Focus on changes the space author can actually make: adding instructions, "
     "example SQL, table/column metadata, join specs, SQL snippets, or rewording "
     "benchmark questions. Be concise and specific."),
    ("human", """Analyse the following benchmark results and provide recommendations.

**Space:** {space_name}
**Description:** {space_description}
**Overall accuracy:** {overall_accuracy:.1f}%
**Runs:** {num_runs}
**Benchmarks:** {num_benchmarks}

---

### Score Reason Breakdown (from Eval API)
{failure_categories}

### Question Consistency
{consistency}

### Per-Question Detail (sorted by pass rate, worst first)
{question_details}

---

The score reasons come from the Genie Eval API and indicate specific failure types:
- LLM_JUDGE_* reasons mean the LLM judge identified a semantic or logical issue in the generated SQL
- RESULT_* reasons mean the output datasets didn't match (missing/extra rows or columns)
- EMPTY_* reasons mean the result or expected SQL was empty
- COLUMN_TYPE_DIFFERENCE means data types didn't align
- SINGLE_CELL_DIFFERENCE means values were close but not matching

Provide your analysis as a structured report with:

1. **Executive Summary** — 2-3 sentence overview of benchmark health and the most critical issues

2. **Root Cause Analysis** — for each major failure pattern, explain the likely cause:
   - Why are certain questions always failing?
   - Why do intermittent failures occur?
   - What do the score reasons reveal about what Genie is getting wrong?

3. **Recommendations** — 5-8 specific, prioritised actions the space author should take.
   For each recommendation:
   - What to change (e.g., "Add an example SQL for...", "Add a text instruction that...")
   - Why it will help
   - Priority: HIGH / MEDIUM / LOW

4. **Quick Wins** — 2-3 changes that are easy to implement and likely to have immediate impact""")
])

analysis_chain = analysis_prompt | llm | StrOutputParser()

failure_categories_text = "\n".join(
    f"  {reason}: {count}" for reason, count in reason_counts.most_common()
) if failure_summary else "  (no failures)"

analysis_report = analysis_chain.invoke({
    "space_name": space.title,
    "space_description": space.description or "No description provided",
    "overall_accuracy": overall_accuracy,
    "num_runs": NUM_RUNS,
    "num_benchmarks": len(benchmark_questions),
    "failure_categories": failure_categories_text,
    "consistency": consistency_text,
    "question_details": per_question_text,
})

print(analysis_report)

## Log Analysis Report to MLflow

Append the LLM-generated analysis report as an artifact to the same MLflow run.

In [ ]:
with mlflow.start_run(run_id=run_id):
    with tempfile.TemporaryDirectory() as tmpdir:
        report_path = os.path.join(tmpdir, "analysis_report.md")
        with open(report_path, "w") as f:
            f.write(analysis_report)
        mlflow.log_artifact(report_path)

print(f"Analysis report logged as artifact to MLflow run {run_id}.")